In [ ]:
### NOTE ###
# The analyses reported in the paper were conducted using Japanese prompts.
# Due to variability in the LLM’s responses, rerunning the analysis will not reproduce exactly the same results.
# The data used in the paper are provided in "results/fig3_labels_va_results_paper.csv".

In [ ]:
from openai import OpenAI
import json
import pandas as pd

In [ ]:
client = OpenAI()
model = "gpt-5.2"

labels_jp = "data/labels_jp.txt"
labels_en = "data/labels_en.txt"

va_results_jp = "results/fig3_labels_va_results_test_jp.csv"
va_results_en = "results/fig3_labels_va_results_test_en.csv"

In [ ]:
# analysis in Japanese (paper)

def create_message(emotion_label):
    system_message="""
    あなたは感情分析の専門家です。
    与えられた感情ラベルのときの、valenceの値とarousalの値を出してください。
    valenceは-1（不快）～1（快）の範囲で出してください。
    arousalは-1（非覚醒）～1（覚醒）の範囲で出してください。
    どちらも小数点以下2桁で答えてください。
    """

    user_message = f"「{emotion_label}」について分析してください。"
    
    input_entry = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ]

    return input_entry

tools = [{
    "type": 'function',
    "name": "analyze_valence_arousal",
    "description": "感情ラベルのvalenceの値及びarousalの値を出力する",
    "parameters": {
        "type": "object",
        "properties": {
            "valence": {"type": "number", "minimum": -1, "maximum": 1,
                        "description": "与えられた感情ラベルのときのvalenceの値。-1が不快で1が快。小数点以下2桁で回答。"},
            "arousal": {"type": "number", "minimum": -1, "maximum": 1,
                        "description": "与えられた感情ラベルのときのarousalの値。-1が非覚醒で1が覚醒。小数点以下2桁で回答。"},
            "reasoning": {"type": 'string',
                          "description": "判断した理由を日本語で1,2文で答えてください。"},
        },
        "required": ["valence", "arousal", "reasoning"],
        "additionalProperties": False
    },
    "strict": True
}]

with open(labels_jp, "r", encoding="utf-8") as f:
    emotion_list = [line.strip() for line in f]

results = []
for num, emotion_label in enumerate(emotion_list):
    print(num, emotion_label)
    input_entry = create_message(emotion_label)

    for i in range(10):
        getresponse=True
        while getresponse:
            try:
                response = client.responses.create(
                    model=model,
                    input=input_entry,
                    tools=tools,
                    tool_choice="required",
                )
                response_output = json.loads(response.output[0].arguments)
                
                results.append({
                    "ID": f"{(num+1):02d}",
                    "emotion": emotion_label,
                    "try_num": f"{i+1}",
                    "valence": response_output["valence"],
                    "arousal": response_output["arousal"],
                    "reasoning": response_output["reasoning"]
                })
                getresponse=False

            except Exception as e:
                print(e)

df = pd.DataFrame(results)
df.to_csv(va_results_jp, index=False, encoding="utf-8-sig")
print("saved:", va_results_jp)

In [ ]:
# analysis in English

def create_message(emotion_label):
    system_message="""
    You are an expert in emotion analysis.
    For the given emotion label, provide valence and arousal values.
    The valence value must range from -1 (unpleasant) to 1 (pleasant).
    The arousal value must range from -1 (low arousal) to 1 (high arousal).
    Provide both values rounded to two decimal places.
    """

    user_message = f"「{emotion_label}」について分析してください。"
    
    input_entry = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ]

    return input_entry

tools = [{
    "type": 'function',
    "name": "analyze_valence_arousal",
    "description": "Output the valence and arousal values for the given emotion label.",
    "parameters": {
        "type": "object",
        "properties": {
            "valence": {"type": "number", "minimum": -1, "maximum": 1,
                        "description": "The valence value for the given emotion label. -1 represents unpleasantness, and 1 represents pleasantness. Provide the value rounded to two decimal places."},
            "arousal": {"type": "number", "minimum": -1, "maximum": 1,
                        "description": "The arousal value for the given emotion label. -1 represents low arousal, and 1 represents high arousal. Provide the value rounded to two decimal places."},
            "reasoning": {"type": 'string',
                          "description": "Explain the reasoning behind your assessment in one or two sentences in English."},
        },
        "required": ["valence", "arousal", "reasoning"],
        "additionalProperties": False
    },
    "strict": True
}]

with open(labels_en, "r", encoding="utf-8") as f:
    emotion_list = [line.strip() for line in f]

results = []
for num, emotion_label in enumerate(emotion_list):
    print(num, emotion_label)
    input_entry = create_message(emotion_label)

    for i in range(10):
        getresponse=True
        while getresponse:
            try:
                response = client.responses.create(
                    model=model,
                    input=input_entry,
                    tools=tools,
                    tool_choice="required",
                )
                response_output = json.loads(response.output[0].arguments)
                
                results.append({
                    "ID": f"{(num+1):02d}",
                    "emotion": emotion_label,
                    "try_num": f"{i+1}",
                    "valence": response_output["valence"],
                    "arousal": response_output["arousal"],
                    "reasoning": response_output["reasoning"]
                })
                getresponse=False

            except Exception as e:
                print(e)

df = pd.DataFrame(results)
df.to_csv(va_results_en, index=False, encoding="utf-8-sig")
print("saved:", va_results_en)